# Fair MFVI experiment notebook

Fit German Credit, Student Performance and ACSIncome models with the current pipeline. Run the notebook from the repository root. Use `python graphs.py` to redraw supported figures from saved results.

Fresh runs are disabled by default. Set `RUN_FRESH` to `True` in the setup cell to enable them. Each run uses a new directory under `runs/`.

The ACS fit uses the current log-normal optimizer. The single-penalty timing cell runs a separate benchmark.


In [ ]:
import os
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"]      = "1"

import sys, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from datetime import datetime

RUN_FRESH = False
REPO_DIR = Path.cwd().resolve()
CODE_DIR = str(REPO_DIR)
DATA_DIR = str(REPO_DIR / "data")
ACS_DATA_DIR = os.getenv("FAIR_MFVI_ACS_DATA_DIR", DATA_DIR)
RUN_DIR = REPO_DIR / "runs" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
RESULTS_DIR = str(RUN_DIR / "results")
FIGURES_DIR = str(RUN_DIR / "figures")
NUM_FOLDS   = 5
ENFORCE     = "test"
SEED        = 42

vec_lambda = np.logspace(-3, 8, 60)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
if RUN_FRESH:
    os.makedirs(RESULTS_DIR)
    os.makedirs(FIGURES_DIR)
print("Fresh fitting is enabled." if RUN_FRESH else "Fresh fitting is disabled.")

In [ ]:
import importlib
import data_loader, cross_validation, vb_logit_fit, vb_logit_pred, optimizer
import load_acs_pvar, comparison
for m in [data_loader, cross_validation, vb_logit_fit, vb_logit_pred,
          optimizer, load_acs_pvar, comparison]:
    importlib.reload(m)

from data_loader      import load_german, load_student
from cross_validation import train_test_split_cv
from optimizer        import run_fold
from vb_logit_pred    import vb_logit_pred
from utils            import logdet
from load_acs_pvar    import (load_acs_income_full, build_design_matrix,
                              total_dummy_columns, CATEGORICAL_COLS_ORDER)

import inspect
print("run_fold:", inspect.signature(run_fold))

In [ ]:
def run_pipeline(tag, X0, y0, isns, only_folds=None):
    """Run CV folds for a dataset and save Results_{tag}_CV_n.npz (no overwrite of originals)."""
    N = len(y0); M = math.floor(N / NUM_FOLDS)
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(N)
    X0s, y0s, isnss = X0[perm], y0[perm], isns[perm]
    folds = list(only_folds) if only_folds is not None else list(range(1, NUM_FOLDS + 1))
    t0 = time.time()
    for n in folds:
        print(f"\n===== {tag} — fold {n}/{NUM_FOLDS} =====")
        X, y, z, XT, yT, zT, *_ = train_test_split_cv(X0s, y0s, isnss, M=M, n=n)
        run_fold(X=X, y=y, z=z, XT=XT, yT=yT, zT=zT,
                 enforce_fairness_on=ENFORCE, vec_lambda=vec_lambda,
                 save_path=f"{RESULTS_DIR}/Results_{tag}_CV_{n}.npz")
    print(f"\n{tag} done in {(time.time()-t0)/60:.1f} min")
    return X0s, y0s, isnss, M


def load_cv(tag):
    """Aggregate per-fold arrays into (n_lam, n_folds) matrices."""
    keys = ["sr_train","sr_test","sr_train_exact","sr_test_exact",
            "p97_train","p2_5_train","p97_test","p2_5_test",
            "dif_train","dif_test"]
    n_lam = len(vec_lambda)
    agg = {k: np.zeros((n_lam, NUM_FOLDS)) for k in keys}
    agg["elbo_raw"] = np.zeros((n_lam, NUM_FOLDS))
    folds = []
    for n in range(1, NUM_FOLDS + 1):
        d = np.load(f"{RESULTS_DIR}/Results_{tag}_CV_{n}.npz", allow_pickle=True)
        for k in keys:
            agg[k][:, n-1] = d[k]
        agg["elbo_raw"][:, n-1] = d["best_L"]
        folds.append(d)
    agg["folds"] = folds
    return agg


def compute_elbo_star(tag, X0s, y0s, isns, M):
    """Recompute a comparable ELBO* across lambdas, fixing (a0,b0,ak,bk) from the first grid entry."""
    n_lam = len(vec_lambda)
    L0 = np.zeros((n_lam, NUM_FOLDS))
    for n in range(NUM_FOLDS):
        d = np.load(f"{RESULTS_DIR}/Results_{tag}_CV_{n+1}.npz", allow_pickle=True)
        betas, sigmas = d["best_beta"], d["best_Sigma"]
        va, vb = d["best_vec_a"], d["best_vec_b"]
        Xf, yf, *_ = train_test_split_cv(X0s, y0s, isns, M=M, n=n+1)
        a0f, b0f = float(va[0][0]),  float(vb[0][0])
        akf, bkf = float(va[0][-1]), float(vb[0][-1])
        for i in range(n_lam):
            if betas[i] is None:
                continue
            beta, Sigma = betas[i].astype(float), sigmas[i].astype(float)
            inv_S = np.linalg.inv(Sigma)
            XSX = (Xf @ Sigma * Xf).sum(axis=1)
            xi = np.sqrt(np.maximum(XSX + (Xf @ beta) ** 2, 0.0))
            with np.errstate(divide="ignore", invalid="ignore"):
                Om = np.tanh(xi / 2.0) / (4.0 * xi)
            Om = np.where(np.isnan(Om), 1.0/8.0, Om)
            Lchi = -np.log1p(np.exp(-xi)) - xi/2.0 + xi**2 * Om
            L0[i, n] = (0.5 * (beta @ inv_S @ beta + logdet(Sigma))
                        + a0f*np.log(b0f) - akf*np.log(bkf)
                        + math.lgamma(akf) - math.lgamma(a0f)
                        + akf - akf*b0f/bkf + Lchi.sum())
    return L0

In [ ]:
def paper_figures(tag):
    """Paper figures with the original aesthetics (fig1 SR+PI, fig2 SR, fig3 unfairness)."""
    r = load_cv(tag)
    lam = vec_lambda; lw = 2; fs = 14
    out = Path(FIGURES_DIR) / tag.lower(); out.mkdir(parents=True, exist_ok=True)
    def save(fig, name):
        for ext in ("pdf", "png"):
            fig.savefig(out / f"{name}.{ext}", bbox_inches="tight", dpi=150)

    sr_tr = r["sr_train"].mean(1); sr_te = r["sr_test"].mean(1)
    p97tr = r["p97_train"].mean(1); p25tr = r["p2_5_train"].mean(1)
    p97te = r["p97_test"].mean(1);  p25te = r["p2_5_test"].mean(1)
    dtr = r["dif_train"].mean(1);   dte = r["dif_test"].mean(1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogx(lam, sr_tr, "r", lw=lw, label="Train Set")
    ax.semilogx(lam, sr_te, "b", lw=lw, label="Test Set")
    ax.semilogx(lam, p97tr, "--r", lw=lw, label="95% PI (Train)")
    ax.semilogx(lam, p25tr, "--r", lw=lw)
    ax.semilogx(lam, p97te, "--b", lw=lw, label="95% PI (Test)")
    ax.semilogx(lam, p25te, "--b", lw=lw)
    ax.set_xlabel(r"$\lambda$", fontsize=24); ax.set_ylabel("Success Rate", fontsize=24)
    ax.tick_params(labelsize=fs); ax.legend(fontsize=16); ax.grid(True, which="both", ls=":")
    fig.tight_layout(); save(fig, f"{tag}_fig1_success_rate_with_PI"); plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogx(lam, sr_tr, "r", lw=lw, label="Train Set")
    ax.semilogx(lam, sr_te, "b", lw=lw, label="Test Set")
    ax.set_xlabel(r"$\lambda$", fontsize=24); ax.set_ylabel("Success Rate", fontsize=24)
    ax.tick_params(labelsize=fs); ax.legend(fontsize=16); ax.grid(True, which="both", ls=":")
    fig.tight_layout(); save(fig, f"{tag}_fig2_success_rate"); plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.loglog(lam, dtr, "--r", lw=lw, label="Train Set")
    ax.loglog(lam, dte, "--b", lw=lw, label="Test Set")
    ax.set_xlabel(r"$\lambda$", fontsize=24); ax.set_ylabel(r"$\mathcal{F}_0$", fontsize=24)
    ax.tick_params(labelsize=fs); ax.legend(fontsize=16); ax.grid(True, which="both", ls=":")
    fig.tight_layout(); save(fig, f"{tag}_fig3_unfairness_metric"); plt.show()


def pareto_figure(tag, X0s, y0s, isns, M):
    """Paper Pareto figure: ELBO* and Success Rate vs unfairness F."""
    r = load_cv(tag)
    L0 = compute_elbo_star(tag, X0s, y0s, isns, M)
    F  = r["dif_test"].mean(1); L = L0.mean(1); sr = r["sr_test"].mean(1)
    v = (F > 0) & np.isfinite(L); order = np.argsort(F[v])
    out = Path(FIGURES_DIR) / tag.lower()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.semilogx(F[v][order], L[v][order], "k", lw=2)
    ax1.set_xlabel(r"$\mathcal{F}$", fontsize=20)
    ax1.set_ylabel(r"$\mathcal{L}^*(q^*, \zeta^*)$", fontsize=18)
    ax1.grid(True, which="both", ls=":"); ax1.tick_params(labelsize=12)
    ax2.semilogx(F[v][order], sr[v][order], "k", lw=2)
    ax2.set_xlabel(r"$\mathcal{F}$", fontsize=20)
    ax2.set_ylabel("Estimated Success Rate", fontsize=14)
    ax2.grid(True, which="both", ls=":"); ax2.tick_params(labelsize=12)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(out / f"{tag}_pareto_front.{ext}", bbox_inches="tight", dpi=150)
    plt.show()

## German

In [ ]:
if RUN_FRESH:
    Xg, yg, isnsg = load_german(f"{DATA_DIR}/german_credit.csv")
    gs = run_pipeline("GERMAN", Xg, yg, isnsg)

In [ ]:
if RUN_FRESH:
    paper_figures("GERMAN")
    pareto_figure("GERMAN", *gs)

In [ ]:
if RUN_FRESH:
    res_german = comparison.run_comparison(
        dataset="german", data_dir=DATA_DIR, results_dir=RESULTS_DIR,
        figures_dir=FIGURES_DIR, num_folds=NUM_FOLDS, seed=SEED)
    comparison.plot_comparison(res_german)

## Student

In [ ]:
if RUN_FRESH:
    Xs, ys, isnss = load_student(f"{DATA_DIR}/student-por.csv")
    ss = run_pipeline("STUDENT", Xs, ys, isnss)

In [ ]:
if RUN_FRESH:
    paper_figures("STUDENT")
    pareto_figure("STUDENT", *ss)

In [ ]:
if RUN_FRESH:
    res_student = comparison.run_comparison(
        dataset="student", data_dir=DATA_DIR, results_dir=RESULTS_DIR,
        figures_dir=FIGURES_DIR, num_folds=NUM_FOLDS, seed=SEED)
    comparison.plot_comparison(res_student)

## ACSIncome (maximum number of predictors)

In [ ]:
if RUN_FRESH:
    df_numeric, dummy_blocks, y0_acs, isns_acs = load_acs_income_full(
        state="NV", year="2018", data_dir=ACS_DATA_DIR)
    n_numeric     = df_numeric.shape[1]
    n_dummy_total = total_dummy_columns(dummy_blocks)
    p_max         = 1 + n_numeric + n_dummy_total
    X0_acs = build_design_matrix(df_numeric, dummy_blocks, n_dummy_total)
    D = X0_acs.shape[1]; N = len(y0_acs)
    tag_acs = f"ACS_NV_P{D}"
    print(f"{tag_acs}  N={N}  p={D}")

In [ ]:
if RUN_FRESH:
    acs = run_pipeline(tag_acs, X0_acs, y0_acs, isns_acs)

In [ ]:
if RUN_FRESH:
    paper_figures(tag_acs)
    pareto_figure(tag_acs, *acs)

### ACSIncome — runtime scaling vs number of predictors

In [ ]:
if RUN_FRESH:
    A0, B0, LAM = 1e-2, 1e-4, 1e4
    p_targets = np.unique(np.round(np.linspace(11, p_max, 25)).astype(int))

    timing = []
    for p_t in p_targets:
        n_dummy = max(0, min(p_t - 1 - n_numeric, n_dummy_total))
        X0 = build_design_matrix(df_numeric, dummy_blocks, n_dummy)
        Dp = X0.shape[1]
        mns = X0[isns_acs].mean(0); ms = X0[~isns_acs].mean(0)
        z = mns - ms; z[0] = 0.0; z = z / np.linalg.norm(z)
        from vb_logit_fit import vb_logit_fit_sigma
        t0 = time.time()
        out = vb_logit_fit_sigma(X0, y0_acs, z, lam=LAM, a0=A0, b0=B0)
        dt = time.time() - t0
        print(f"p={Dp:4d}  time={dt:7.2f}s")
        timing.append({"p": Dp, "time_s": dt})
    df_timing = pd.DataFrame(timing)
    df_timing.to_csv(f"{RESULTS_DIR}/{tag_acs}_timing_vs_p.csv", index=False)
    df_timing

In [ ]:
if RUN_FRESH:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df_timing["p"], df_timing["time_s"], marker="o", lw=2)
    ax.set_xlabel("p (number of predictors)", fontsize=16)
    ax.set_ylabel("CAVI runtime (s)", fontsize=16)
    ax.grid(True, which="both", ls=":")
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(f"{FIGURES_DIR}/{tag_acs}_timing_vs_p.{ext}", bbox_inches="tight", dpi=150)
    plt.show()